# 02 — Baseline Two-Model (T-Learner) + The Trap Demo

**The core argument of this project, in one figure:**
A model optimized for outcome ROC-AUC ranks users very differently from a model ranked by
estimated uplift — and the AUC-optimal ranking is *worse* on the Qini curve.

This is the "wrong metric" trap: persuadables ≠ high-converters.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

from src.data import load_data, split, get_Xyt, FEATURE_COLS
from src.learners import TLearner
from src.metrics import qini_curve, qini_auc, qini_coefficient, uplift_at_k, evaluate_all

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

In [ ]:
df = load_data(percent10=True)
train, val, test = split(df)
X_tr, y_tr, t_tr = get_Xyt(train)
X_te, y_te, t_te = get_Xyt(test)
print(f'Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}')

## 1. T-learner (two-model) uplift

In [ ]:
tl = TLearner()
tl.fit(X_tr, y_tr, t_tr)
uplift_tl = tl.predict(X_te)
print('T-learner metrics:')
print(evaluate_all(y_te, uplift_tl, t_te, 'T-learner'))

## 2. The Trap: ROC-AUC–optimal classifier as a "targeting" model

Train a classifier on all users (both arms) to predict `visit`. Use `P(visit|X)` as a
targeting score. This is what a naive ML team would do.

In [ ]:
clf_trap = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63,
                           n_jobs=-1, verbose=-1, random_state=42)
clf_trap.fit(X_tr, y_tr)
prob_visit = clf_trap.predict_proba(X_te)[:, 1]

auc_trap = roc_auc_score(y_te, prob_visit)
print(f'ROC-AUC of outcome classifier: {auc_trap:.4f}  (looks great...)')
print()
print('But what is its Qini coefficient?')
print(evaluate_all(y_te, prob_visit, t_te, 'P(visit) classifier'))

## 3. The Key Figure: Qini curve comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# -- Left: Qini curves
ax = axes[0]
for name, scores in [('T-learner (uplift)', uplift_tl), ('P(visit) classifier', prob_visit)]:
    x, q = qini_curve(y_te, scores, t_te)
    frac = x / len(y_te)
    ax.plot(frac, q, label=f'{name}  Qini={qini_coefficient(y_te, scores, t_te):.3f}')

# random baseline
_, q_tl = qini_curve(y_te, uplift_tl, t_te)
frac_all = np.linspace(0, 1, len(y_te))
ax.plot([0, 1], [0, q_tl[-1]], 'k--', linewidth=0.8, label='Random baseline')
ax.set_xlabel('Fraction of population treated')
ax.set_ylabel('Qini score')
ax.set_title('Qini curves: uplift model vs outcome classifier')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# -- Right: rank correlation scatter (sample for speed)
ax2 = axes[1]
idx = np.random.choice(len(y_te), min(5000, len(y_te)), replace=False)
ax2.scatter(prob_visit[idx], uplift_tl[idx], alpha=0.15, s=3, color='#2ca02c')
from scipy.stats import pearsonr, spearmanr
rho, _ = spearmanr(prob_visit, uplift_tl)
ax2.set_xlabel('P(visit) score (outcome classifier)')
ax2.set_ylabel('Predicted uplift (T-learner)')
ax2.set_title(f'Rank correlation of scoring methods\nSpearman ρ = {rho:.3f}')
ax2.grid(alpha=0.3)

plt.suptitle('The Trap: High ROC-AUC ≠ Good Uplift Targeting', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/trap_demo.png', bbox_inches='tight')
plt.show()

print('\nKey insight: The AUC-optimal classifier ranks users by BASELINE conversion,')
print('not by INCREMENTAL lift. Sure Things (high baseline, zero uplift) dominate its top decile.')
print('The T-learner, despite lower AUC, produces better TARGETING.')

## 4. The Four-Quadrant Mental Model

In [ ]:
# Approximate quadrant assignments on test set
median_prob = np.median(prob_visit)
median_uplift = 0  # uplift centered near 0

persuadables   = ((prob_visit < median_prob) & (uplift_tl > median_uplift)).sum()
sure_things    = ((prob_visit > median_prob) & (uplift_tl < median_uplift)).sum()
lost_causes    = ((prob_visit < median_prob) & (uplift_tl < median_uplift)).sum()
sleeping_dogs  = ((uplift_tl < 0)).sum()

print('Approximate four-quadrant population breakdown:')
print(f'  Persuadables  (low baseline, high uplift): {persuadables:,} ({persuadables/len(y_te):.1%})')
print(f'  Sure Things   (high baseline, low uplift): {sure_things:,} ({sure_things/len(y_te):.1%})')
print(f'  Lost Causes   (low baseline, low uplift):  {lost_causes:,} ({lost_causes/len(y_te):.1%})')
print(f'  Sleeping Dogs (negative predicted uplift):  {sleeping_dogs:,} ({sleeping_dogs/len(y_te):.1%})')
print()
print('The existence of Sleeping Dogs is why uplift ≠ propensity: advertising HURTS some users.')